<a href="https://colab.research.google.com/github/harsh-github007/Identifying-Toxicity-Within-Esports-YouTube-Channels/blob/main/notebooks/run_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Toxicity in Indian esports YouTube comments

Runs the whole study: collect comments, score them, label a sample by hand, then measure and report.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.* Scoring is about 10× faster on a GPU.
2. Have a YouTube Data API key ready (free; see the README).

Steps 1–4 take about 15 minutes. Step 5 is the hand labelling, then step 6 produces the results.

In [1]:
# 1. Get the code
!git clone -q https://github.com/harsh-github007/Identifying-Toxicity-Within-Esports-YouTube-Channels.git repo
%cd repo
!pip install -q -e ".[model]"

/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for esports-toxicity (pyproject.toml) ... done


In [2]:
# 2. API key: stored as a Colab secret named YOUTUBE_API_KEY (key icon in the left sidebar), or typed here
import os, getpass
try:
    from google.colab import userdata
    os.environ["YOUTUBE_API_KEY"] = userdata.get("YOUTUBE_API_KEY")
except Exception:
    os.environ["YOUTUBE_API_KEY"] = getpass.getpass("YouTube API key: ")

YouTube API key: ··········


In [3]:
# 3. Collect comments (edit config/channels.yaml first if a handle needs fixing)
!python -m toxicity collect

  Total Gaming      10 videos    5057 comments
  Techno Gamerz     10 videos    5077 comments
  Gyan Gaming       10 videos     342 comments
  A_S Gaming         1 videos       0 comments
  Desi Gamers        0 videos       0 comments
Saved 10,471 comments to data/comments.csv


In [4]:
# 4. Score every comment with the model and the word list, then draw the labelling sample
!python -m toxicity score
!python -m toxicity sample -n 400
from google.colab import files
try:
    files.download("data/annotation/to_label.csv")
except Exception:
    pass
print("If no download started: open the Files panel (folder icon, left), go to repo/data/annotation,\n"
      "and right-click to_label.csv → Download.")

Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.4-alpha/multilingual_debiased-0b549669.ckpt" to /root/.cache/torch/hub/checkpoints/multilingual_debiased-0b549669.ckpt
100% 1.04G/1.04G [00:48<00:00, 23.0MB/s]
config.json: 100% 615/615 [00:00<00:00, 2.79MB/s]
Loading weights: 100% 201/201 [00:00<00:00, 266.56it/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 115kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 4.49MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:01<00:00, 8.31MB/s]
  scored 64 / 10,471
  scored 1,344 / 10,471
  scored 2,624 / 10,471
  scored 3,904 / 10,471
  scored 5,184 / 10,471
  scored 6,464 / 10,471
  scored 7,744 / 10,471
  scored 9,024 / 10,471
  scored 10,304 / 10,471
Scored 10,471 comments -> data/scored.csv
Label the 'toxic' column (1 or 0) in data/annotation/to_label.csv, then save it as data/annotation/labels.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

If no download started: open the Files panel (folder icon, left), go to repo/data/annotation,
and right-click to_label.csv → Download.


## 5. Label the sample

Open `to_label.csv` in Google Sheets or Excel and fill the **toxic** column with `1` or `0` for every row,
following [docs/labelling-guide.md](https://github.com/harsh-github007/Identifying-Toxicity-Within-Esports-YouTube-Channels/blob/main/docs/labelling-guide.md).
The optional **category** column takes `insult`, `profanity`, `threat` or `hate`.

The sheet doesn't show which channel a comment came from or what the model thought, so the labels stay independent.
Save it as CSV, then upload it with the next cell.

In [ ]:
# Upload your labelled file
import pandas as pd
from google.colab import files
up = files.upload()
name = next(iter(up))
os.replace(name, "data/annotation/labels.csv")
lab = pd.read_csv("data/annotation/labels.csv", dtype=str, sep=None, engine="python", encoding="utf-8-sig")
done = lab["toxic"].fillna("").str.strip().ne("").sum()
print(f"{done} of {len(lab)} comments labelled ({(lab['toxic'].str.strip() == '1').sum()} toxic)")
if done < 50:
    print("Label more rows before running the next cell. At least 50 are needed; all 400 is best.")


In [ ]:
# 6. Measure the methods, estimate toxicity per channel, draw the charts
!python -m toxicity evaluate
import os
from IPython.display import Markdown, Image, display
if os.path.exists("results/results.md"):
    display(Markdown(open("results/results.md").read().replace("figures/", "results/figures/")))
    display(Image("results/figures/prevalence.png")); display(Image("results/figures/methods.png"))
else:
    print("No results yet. Fix the message above, then run this cell again.")


In [ ]:
# 7. Download the results to add to the repository (results/ and your labels)
!zip -qr results_to_commit.zip results data/annotation/labels.csv data/annotation/sample_key.csv config
files.download("results_to_commit.zip")